# Exp7 — Eval All Router Sizes v7 (Colab A100)

Runs FP16 inference on 380 val v7 samples for v7 router models.

**v7 schema**: multi-turn ChatML (history list + input + output 4-keys core).

**Metrics** (extends v6 evaluation):
- Routing accuracy = intent + scope both correct (primary)
- Intent accuracy + Wilson 95% CI
- Scope accuracy
- Macro-F1 + per-intent F1
- Per-intent accuracy + top confusion pairs
- **Multi-turn routing acc** (turn=2/3 only) — anaphora resolution check
- **Rewrite quality**: exact-match + entity preservation (ward/district mention from history)
- **Confidence calibration ECE** (5-tier scheme alignment)
- **Tier-conditional accuracy** (T1/T2/T3/T4/T5 routing acc per tier)
- Latency P50/P90/P95

**Output**: `exp7_summary.json` + plots + per-sample CSV.

In [ ]:
%%capture
!pip install -U transformers accelerate huggingface_hub scikit-learn


In [ ]:
import os, json, time, re, gc, math
import torch
from pathlib import Path
from collections import Counter, defaultdict
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Set HF_TOKEN in Colab Secrets'
login(token=HF_TOKEN)

import subprocess
gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader']).decode().strip()
print(f'GPU: {gpu}')


In [ ]:
# ═══════════════════════ CONFIGS (v7) ═══════════════════════
HF_DATA_REPO = 'daredevil467/hanoi-weather-router-data-v7'

MODELS = [
    ('Qwen3-4B-v7',  'daredevil467/hanoi-router-qwen3-4b-v7-1',  4.0),
    ('Qwen3-8B-v7',  'daredevil467/hanoi-router-qwen3-8b-v7',  8.0),
]
# Optionally add v6 baselines for comparison (uncomment):
# MODELS += [
#     ('Qwen3-4B-v6 (baseline)',  'daredevil467/hanoi-router-qwen3-4b-v6',  4.0),
# ]

OUTPUT_DIR = Path('/content/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ═══════════════════════ LOAD VAL DATA + SYSTEM PROMPT (v7) ═══════════════════════
val_file    = hf_hub_download(repo_id=HF_DATA_REPO, filename='multitask_val.jsonl', repo_type='dataset')
prompt_file = hf_hub_download(repo_id=HF_DATA_REPO, filename='system_prompt.txt',   repo_type='dataset')

with open(val_file, encoding='utf-8') as f:
    val_samples = [json.loads(l) for l in f]
with open(prompt_file, encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read().strip()

print(f'Val samples: {len(val_samples)}')
print(f'System prompt: {len(SYSTEM_PROMPT)} chars')

# v7 distribution checks
from collections import Counter
turn_dist = Counter(len(s['history']) + 1 for s in val_samples)
print(f'Turn distribution: {dict(turn_dist)}')
n_with_history = sum(1 for s in val_samples if s['history'])
n_with_rw      = sum(1 for s in val_samples if s['output'].get('rewritten_query'))
print(f'With history (multi-turn): {n_with_history}  |  With rewrite: {n_with_rw}')

In [ ]:
# ═══════════════════════ HELPERS (v7 multi-turn) ═══════════════════════

def wilson_ci(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    margin = (z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / denom
    return (round((center - margin)*100, 2), round((center + margin)*100, 2))

JSON_RE = re.compile(r'\{[^{}]*\}', re.DOTALL)

def parse_output(text: str):
    """Parse JSON from model output, stripping </think> tags (Qwen3)."""
    if '</think>' in text:
        text = text[text.rfind('</think>') + len('</think>'):].strip()
    m = JSON_RE.search(text)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def build_prompt(sample: dict) -> str:
    """v7 multi-turn ChatML — must match training format exactly."""
    history = sample.get('history', []) or []
    user_msg = str(sample.get('input', '')).strip()
    text = '<|im_start|>system\n' + SYSTEM_PROMPT + '<|im_end|>\n'
    for h in history:
        text += '<|im_start|>user\n'      + h['user']      + '<|im_end|>\n'
        text += '<|im_start|>assistant\n' + h['assistant'] + '<|im_end|>\n'
    text += '<|im_start|>user\n' + user_msg + '<|im_end|>\n<|im_start|>assistant\n'
    return text

def expected_calibration_error(probs, correct, n_bins=10):
    """ECE for confidence calibration. Returns ECE in [0,1] (lower is better)."""
    bins = [(i/n_bins, (i+1)/n_bins) for i in range(n_bins)]
    ece = 0.0
    n = len(probs)
    for lo, hi in bins:
        in_bin = [(p, c) for p, c in zip(probs, correct) if lo < p <= hi]
        if not in_bin: continue
        bin_acc = sum(c for _, c in in_bin) / len(in_bin)
        bin_conf = sum(p for p, _ in in_bin) / len(in_bin)
        ece += (len(in_bin)/n) * abs(bin_acc - bin_conf)
    return ece

In [ ]:
# ═══════════════════════ EVAL LOOP — v7 multi-turn aware ═══════════════════════
# Bugs fixed (vs v6 template):
# - rw_entity_ok now extracts admin entity from EXPECTED rewrite (regex Phường|Xã|Quận|Huyện X)
#   instead of v6 sample['context']['location'] (v7 doesn't have context field).
# - has_context replaced by has_history.
# Plus new v7 metrics: tier-conditional acc, turn-conditional acc, 5-tier ECE.
from transformers import AutoModelForCausalLM, AutoTokenizer

# Helper: extract first admin entity phrase from text ("Phường Cầu Giấy", "Xã Minh Châu"...)
ENTITY_RE = re.compile(
    r'(Phường|Xã|Quận|Huyện|Thị xã)\s+[A-Z\u00C0-\u1EF9][\w\s\-\u00C0-\u1EF9]*?'
    r'(?=[\s,\.\?\!\(\)]|$|hôm|hiện|bây|sáng|chiều|tối|trưa|đêm|tuần|tháng|ngày|năm|gần|mùa|lúc|từ|đến|và|có|không|nóng|lạnh|mát|gió|mưa|đang|còn|nên|sẽ|cảnh|báo|thì|thế|nhỉ|nào|trong|so|với)',
    re.UNICODE
)

def extract_entity(text):
    if not text: return None
    m = ENTITY_RE.search(text)
    return m.group(0).strip() if m else None

results = {}
per_sample_rows = []

for name, repo, params_b in MODELS:
    print(f'\n{"="*60}')
    print(f'Evaluating {name} ({repo})')
    print(f'{"="*60}')

    tokenizer = AutoTokenizer.from_pretrained(repo, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        repo, token=HF_TOKEN, torch_dtype=torch.float16, device_map='auto',
    )
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ── Counters ─────────────────────────────────────────────────────────
    correct_route  = 0
    total_route    = 0
    correct_intent = 0
    correct_scope  = 0
    parse_failures = 0
    # Rewrite tracking (v7 schema)
    rw_correct, rw_total, rw_entity_ok = 0, 0, 0
    norw_correct, norw_total = 0, 0
    # Per-intent
    intent_correct = Counter()
    intent_total   = Counter()
    confusion_pairs = Counter()
    intent_true, intent_pred = [], []
    latencies = []
    # Tier-conditional (5-tier confidence)
    tier_correct = Counter()  # by expected confidence
    tier_total   = Counter()
    # Turn-conditional (1 / 2 / 3)
    turn_correct = Counter()
    turn_total   = Counter()
    # ECE — predicted confidence vs correctness
    ece_probs = []     # predicted confidence
    ece_correct = []   # 1 if route_ok else 0

    for idx, sample in enumerate(val_samples):
        prompt = build_prompt(sample)
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=4096).to('cuda')
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.eos_token_id,
            )
        latency_ms = (time.time() - t0) * 1000
        latencies.append(latency_ms)

        gen_ids = out[0][inputs['input_ids'].shape[1]:]
        gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        pred = parse_output(gen_text)

        gt = sample['output']
        expected_intent = gt['intent']
        expected_scope  = gt['scope']
        expected_conf   = round(float(gt.get('confidence', 0.85)), 2)
        expected_rw     = gt.get('rewritten_query')
        has_rw          = bool(expected_rw)
        has_history     = bool(sample.get('history'))
        turn_n          = len(sample.get('history', []) or []) + 1

        intent_true.append(expected_intent)
        intent_total[expected_intent] += 1
        total_route += 1
        tier_total[expected_conf] += 1
        turn_total[turn_n] += 1

        if pred is None:
            parse_failures += 1
            intent_pred.append('<parse_error>')
            confusion_pairs[(expected_intent, '<parse_error>')] += 1
            if has_rw: rw_total += 1
            else:      norw_total += 1
        else:
            pi = pred.get('intent', '')
            ps = pred.get('scope',  '')
            pc = round(float(pred.get('confidence', 0.85)), 2)
            intent_pred.append(pi)
            route_ok = (pi == expected_intent and ps == expected_scope)

            if pi == expected_intent:
                correct_intent += 1
                intent_correct[expected_intent] += 1
            else:
                confusion_pairs[(expected_intent, pi)] += 1
            if ps == expected_scope:
                correct_scope += 1
            if route_ok:
                correct_route += 1
                tier_correct[expected_conf] += 1
                turn_correct[turn_n] += 1

            # ECE — uses PREDICTED confidence
            ece_probs.append(pc)
            ece_correct.append(1 if route_ok else 0)

            # Rewrite tracking (v7 fix: extract entity from EXPECTED rw, check pred rw contains it)
            if has_rw:
                rw_total += 1
                if route_ok:
                    rw_correct += 1
                pred_rw = pred.get('rewritten_query') or ''
                expected_entity = extract_entity(expected_rw)
                if expected_entity and expected_entity.lower() in pred_rw.lower():
                    rw_entity_ok += 1
            else:
                norw_total += 1
                if route_ok:
                    norw_correct += 1

        per_sample_rows.append({
            'model': name, 'query': sample['input'],
            'has_history': has_history,                       # v7 fix
            'turn': turn_n,
            'gt_intent': expected_intent, 'gt_scope': expected_scope,
            'gt_conf': expected_conf,
            'pred_raw': gen_text,
            'pred_intent': intent_pred[-1],
            'pred_scope': pred.get('scope', '') if pred else '',
            'pred_conf': pred.get('confidence', None) if pred else None,
            'route_ok': pred is not None and (pred.get('intent') == expected_intent and pred.get('scope') == expected_scope),
            'latency_ms': round(latency_ms, 1),
        })

        if (idx + 1) % 50 == 0:
            print(f'  [{idx+1}/{len(val_samples)}] routing_acc={correct_route/(idx+1)*100:.1f}%')

    # ── Compute metrics ──
    n = len(val_samples)
    routing_acc = correct_route / n
    intent_acc  = correct_intent / n
    scope_acc   = correct_scope  / n
    ci_low, ci_high = wilson_ci(correct_route, n)
    ci_intent_low, ci_intent_high = wilson_ci(correct_intent, n)

    from sklearn.metrics import f1_score, classification_report
    labels = sorted(set(intent_true))
    macro_f1 = f1_score(intent_true, intent_pred, labels=labels, average='macro', zero_division=0)
    per_intent_report = classification_report(intent_true, intent_pred, labels=labels, zero_division=0, output_dict=True)

    sorted_lats = sorted(latencies)
    def pctl(p): return sorted_lats[int(len(sorted_lats)*p)] if sorted_lats else 0

    # ECE
    ece = expected_calibration_error(ece_probs, ece_correct, n_bins=10) if ece_probs else 0.0

    results[name] = {
        'repo': repo,
        'params_billions': params_b,
        'routing_accuracy_pct':  round(routing_acc*100, 2),
        'routing_ci95':          [ci_low, ci_high],
        'intent_accuracy_pct':   round(intent_acc*100, 2),
        'intent_ci95':           [ci_intent_low, ci_intent_high],
        'scope_accuracy_pct':    round(scope_acc*100, 2),
        'macro_f1':              round(macro_f1, 4),
        'parse_failures':        parse_failures,
        # Rewrite metrics (v7 fixed)
        'rewrite_routing_acc':   round(rw_correct/rw_total*100, 2) if rw_total else None,
        'rewrite_entity_cov':    round(rw_entity_ok/rw_total*100, 2) if rw_total else None,
        'rewrite_n':             rw_total,
        'no_rewrite_routing_acc': round(norw_correct/norw_total*100, 2) if norw_total else None,
        # Tier-conditional (5-tier)
        'tier_conditional_acc': {
            f'{conf:.2f}': {
                'n': tier_total[conf],
                'correct': tier_correct[conf],
                'acc': round(tier_correct[conf]/tier_total[conf]*100, 2) if tier_total[conf] else None
            } for conf in sorted(tier_total)
        },
        # Turn-conditional
        'turn_conditional_acc': {
            f'turn_{t}': {
                'n': turn_total[t],
                'correct': turn_correct[t],
                'acc': round(turn_correct[t]/turn_total[t]*100, 2) if turn_total[t] else None,
            } for t in sorted(turn_total)
        },
        # Calibration
        'calibration_ece':       round(ece, 4),
        # Latency
        'latency_p50_ms':        round(pctl(0.50), 1),
        'latency_p90_ms':        round(pctl(0.90), 1),
        'latency_p95_ms':        round(pctl(0.95), 1),
        # Per-intent detail
        'per_intent_f1':         {k: round(v['f1-score'], 3) for k, v in per_intent_report.items() if isinstance(v, dict) and 'f1-score' in v},
        'per_intent_accuracy':   {i: round(intent_correct[i]/intent_total[i]*100, 1) if intent_total[i] else 0 for i in sorted(intent_total)},
        'top_confusion_pairs':   [(f'{e}->{p}', c) for (e, p), c in confusion_pairs.most_common(10)],
    }

    # ── Print summary ──
    print(f'  Routing acc:   {correct_route}/{n} = {routing_acc:.1%}  CI=[{ci_low}, {ci_high}]')
    print(f'  Intent acc:    {correct_intent}/{n} = {intent_acc:.1%}  Macro-F1={macro_f1:.4f}')
    print(f'  Scope acc:     {correct_scope}/{n} = {scope_acc:.1%}')
    if rw_total:
        print(f'  Rewrite routing: {rw_correct}/{rw_total} = {rw_correct/rw_total:.1%}')
        print(f'  Rewrite entity cov: {rw_entity_ok}/{rw_total} = {rw_entity_ok/rw_total:.1%}  [v7-fixed]')
    print(f'  ECE (5-tier):  {ece:.4f}')
    print(f'  P50={pctl(0.50):.0f}ms  ParseFail={parse_failures}')

    # Tier-conditional
    print(f'\n  Tier-conditional accuracy:')
    print(f'  {"Tier":<8} {"Conf":<6} {"Correct":>8} {"Total":>6} {"Acc":>7}')
    tier_label = {0.92:'T1', 0.85:'T2', 0.80:'T3', 0.74:'T4', 0.62:'T5'}
    for c in sorted(tier_total, reverse=True):
        t = tier_total[c]
        cc = tier_correct.get(c, 0)
        acc = cc/t if t else 0
        print(f'  {tier_label.get(c, "?"):<8} {c:<6} {cc:>8} {t:>6} {acc:>6.1%}')

    # Turn-conditional
    print(f'\n  Turn-conditional accuracy:')
    for t in sorted(turn_total):
        n_t = turn_total[t]
        c_t = turn_correct.get(t, 0)
        acc = c_t/n_t if n_t else 0
        flag = ' ←multi-turn' if t > 1 else ''
        print(f'  turn={t}: {c_t}/{n_t} = {acc:.1%}{flag}')

    # Per-intent
    print(f'\n  Per-intent accuracy:')
    print(f'  {"Intent":<25} {"Correct":>7} {"Total":>7} {"Acc":>7}')
    for intent in sorted(intent_total):
        t = intent_total[intent]
        c = intent_correct.get(intent, 0)
        acc = c/t if t else 0
        flag = ' <<<' if acc < 0.85 else ''
        print(f'  {intent:<25} {c:>7} {t:>7} {acc:>6.1%}{flag}')

    if confusion_pairs:
        print(f'\n  TOP CONFUSION:')
        for (exp, pred), cnt in confusion_pairs.most_common(5):
            print(f'    {exp:<25} -> {pred:<25} x{cnt}')

    # Free GPU
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# ═══════════════════════ SAVE RESULTS ═══════════════════════
summary = {
    'experiment': 'Exp6_Router_Size_Ablation',
    'rq': 'RQ5: Is Qwen3-4B the sweet spot for router size?',
    'dataset': 'multitask_val_v5_clean.jsonl (373 samples, 0% leakage)',
    'inference_precision': 'fp16 (merged_16bit)',
    'primary_metric': 'routing_accuracy_pct (intent+scope both correct, same as v4)',
    'models': results,
}

with open(OUTPUT_DIR / 'exp6_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# Per-sample CSV
import csv
with open(OUTPUT_DIR / 'exp6_per_sample.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=per_sample_rows[0].keys())
    w.writeheader()
    w.writerows(per_sample_rows)

print('Saved:')
print(f'  {OUTPUT_DIR}/exp6_summary.json')
print(f'  {OUTPUT_DIR}/exp6_per_sample.csv')


In [ ]:
# ═══════════════════════ PLOTS ═══════════════════════
import matplotlib.pyplot as plt

sizes = [results[name]['params_billions'] for name, _, _ in MODELS]
raccs = [results[name]['routing_accuracy_pct'] for name, _, _ in MODELS]
iaccs = [results[name]['intent_accuracy_pct'] for name, _, _ in MODELS]
lats  = [results[name]['latency_p50_ms'] for name, _, _ in MODELS]
names = [name for name, _, _ in MODELS]

# Figure 1: Size vs Routing Accuracy (primary, v4-aligned)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sizes, raccs, 'o-', linewidth=2, markersize=10, label='Routing (intent+scope)')
ax.plot(sizes, iaccs, 's--', linewidth=1.5, markersize=8, alpha=0.6, label='Intent only')
for s, a, n in zip(sizes, raccs, names):
    ax.annotate(f'{n}\n{a:.1f}%', (s, a), textcoords='offset points', xytext=(8, -4), fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('Model size (billion params, log scale)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Exp6: Router Size vs Routing Accuracy (373 clean val)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp6_size_vs_accuracy.png', dpi=150)
plt.show()

# Figure 2: Size vs Latency
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sizes, lats, 's-', color='orange', linewidth=2, markersize=10)
for s, l, n in zip(sizes, lats, names):
    ax.annotate(n, (s, l), textcoords='offset points', xytext=(8, -4), fontsize=9)
ax.set_xscale('log')
ax.set_xlabel('Model size (billion params, log scale)')
ax.set_ylabel('Latency P50 (ms, GPU FP16)')
ax.set_title('Exp6: Router Size vs Inference Latency')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp6_size_vs_latency.png', dpi=150)
plt.show()

# Figure 3: Per-intent F1 heatmap
import numpy as np
intents = sorted(results[MODELS[0][0]]['per_intent_f1'].keys())
f1_matrix = []
for name, _, _ in MODELS:
    f1_matrix.append([results[name]['per_intent_f1'].get(i, 0) for i in intents])
f1_matrix = np.array(f1_matrix)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(f1_matrix, cmap='RdYlGn', vmin=0.5, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(intents)))
ax.set_xticklabels(intents, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(MODELS)))
ax.set_yticklabels(names)
for i in range(len(MODELS)):
    for j in range(len(intents)):
        ax.text(j, i, f'{f1_matrix[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax, label='F1')
ax.set_title('Exp6: Per-Intent F1 by Router Size')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp6_per_intent_f1_heatmap.png', dpi=150)
plt.show()

print('\nDownload all files from /content/outputs/ (left panel > Files)')
